In [23]:
import pandas as pd
data = pd.read_json("C:\\Users\\Tereza\\Desktop\\PROJETO TCC\\assistente_IA\\DATASET\\annotations_desafio\\val.json")

In [24]:
data.head()

,flaws,image,unrecognizable
0,"{'FRM': 2, 'BLR': 0, 'DRK': 0, 'BRT': 0, 'OBS'...",VizWiz_val_00002854.jpg,0
1,"{'FRM': 0, 'BLR': 0, 'DRK': 0, 'BRT': 0, 'OBS'...",VizWiz_val_00001607.jpg,0
2,"{'FRM': 0, 'BLR': 3, 'DRK': 0, 'BRT': 0, 'OBS'...",VizWiz_val_00005141.jpg,1
3,"{'FRM': 4, 'BLR': 5, 'DRK': 0, 'BRT': 0, 'OBS'...",VizWiz_val_00000516.jpg,0
4,"{'FRM': 3, 'BLR': 0, 'DRK': 0, 'BRT': 0, 'OBS'...",VizWiz_val_00007258.jpg,0


In [25]:
categorias = ["BLR", "FRM", "DRK", "BRT"]

for categoria in categorias:
    globals()[f"df_{categoria.lower()}"] = pd.DataFrame({
        categoria: data["flaws"].apply(lambda x: x[categoria])
    })

In [26]:
df_blr.head(8)

,BLR
0,0
1,0
2,3
3,5
4,0
5,0
6,0
7,3


contagem de 'sim' de cada imagem, de 0 a 5. representa quantos avaliadores disseram que ela estava desfocada (para BLR)

In [27]:
acordo_blr = 1 - df_blr["BLR"] * (5 - df_blr["BLR"]) / 10
acordo_blr.head(8)

0    1.0
1    1.0
2    0.4
3    1.0
4    1.0
5    1.0
6    1.0
7    0.4
Name: BLR, dtype: float64

acordo observado: média do acordo de todas as imagens (o que os avaliadores de fato fizeram)

In [28]:
observado_blr = acordo_blr.mean()
observado_blr

np.float64(0.7468387096774194)

acordo esperado: quanto os avaliadores concordariam por acaso, só pela proporção de 'sim' na categoria inteira (p). dois votos sorteados concordam se ambos são 'sim' (p²) ou ambos 'não' ((1 - p)²)

In [29]:
n_imagens = len(df_blr)
p_sim_blr = df_blr["BLR"].sum() / (n_imagens * 5)

esperado_blr = p_sim_blr**2 + (1 - p_sim_blr)**2
p_sim_blr, esperado_blr

(np.float64(0.32216774193548386), np.float64(0.5632486240166494))

In [30]:
kappa_blr = (observado_blr - esperado_blr) / (1 - esperado_blr)
kappa_blr

np.float64(0.42035376590952883)

as mesmas etapas do blr (acordo por imagem, observado, esperado, kappa) reunidas numa função, para repetir nas outras categorias sem copiar o bloco. o número de avaliadores (5) é um parâmetro; o total de pares sai dele (n × (n - 1) / 2)

In [31]:
def kappa_fleiss(votos_sim, n_anotadores=5):
    pares_total = n_anotadores * (n_anotadores - 1) / 2
    pares_discordantes = votos_sim * (n_anotadores - votos_sim)
    observado = (1 - pares_discordantes / pares_total).mean()

    p_sim = votos_sim.sum() / (len(votos_sim) * n_anotadores)
    esperado = p_sim**2 + (1 - p_sim)**2

    return {
        "kappa": (observado - esperado) / (1 - esperado),
        "observado": observado,
        "esperado": esperado,
        "p_sim": p_sim,
    }

tabela das quatro categorias na validação. kappa = acordo além do acaso; observado = acordo médio entre os avaliadores; esperado = acordo só pela frequência de 'sim'; p_sim = proporção de votos 'sim' na categoria. o kappa mede consistência entre avaliadores, não se eles estão certos

In [32]:
resultados = pd.DataFrame(
    {
        categoria: kappa_fleiss(data["flaws"].apply(lambda x: x[categoria]))
        for categoria in categorias
    }
).T
resultados.round(3)

,kappa,observado,esperado,p_sim
BLR,0.420,0.747,0.563,0.322
FRM,0.231,0.636,0.527,0.384
DRK,0.254,0.913,0.884,0.062
BRT,0.222,0.914,0.890,0.059
